<a href="https://colab.research.google.com/github/paulpdelancy-spec/gdp-dashboard/blob/main/Another_copy_of_Untitled27.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai
print("✅ google-genai installed")



# ═══════════════════════════════════════════════════════
# COMPLETE AI REPORTS SYSTEM — Single Cell Version
# Run this after Cell 1 every session
# ═══════════════════════════════════════════════════════
print("⏳ Loading Complete System...")

# ── IMPORTS ───────────────────────────────────────────
from google.colab import userdata
from google import genai
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timezone
import time
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

# ── CREDENTIALS ───────────────────────────────────────
GEMINI_API_KEY  = userdata.get('GEMINI_API_KEY')
SENDER_PASSWORD = userdata.get('GMAIL_APP_PASSWORD')
SENDER_EMAIL    = "paul.pdelancy@gmail.com"
CONTACT_EMAIL_1 = "paulacumberbatch@yahoo.com"
CONTACT_EMAIL_2 = "paul.pdelancy@gmail.com"
STRIPE_MONTHLY  = "https://buy.stripe.com/9B66oIcTW85N16o4xe8g00a"
STRIPE_YEARLY   = "https://buy.stripe.com/7sY5kE5rufyf9CUe7O8g00b"

# ── GEMINI CLIENT ─────────────────────────────────────
client = genai.Client(api_key=GEMINI_API_KEY)

# ── SUBSCRIPTION CONFIG ───────────────────────────────
SUBSCRIPTION_MONTHLY = 10.00
SUBSCRIPTION_YEARLY  = 120.00
REPORTS_PER_DAY      = 4

# ── SUBSCRIBERS ───────────────────────────────────────
SUBSCRIBERS = [
    ("paulacumberbatch@yahoo.com", "Paul Cumberbatch", 4),
]

# ── SPORTS SITES ──────────────────────────────────────
SPORTS_SITES = [
    {"url": "https://www.afa.com.ar/es/",                "league": "AFA Argentina",         "lang": "es"},
    {"url": "https://dimayor.com.co/",                   "league": "Dimayor Colombia",       "lang": "es"},
    {"url": "https://www.cfl.ca/",                       "league": "CFL Canada",             "lang": "en"},
    {"url": "https://www.mlb.com/",                      "league": "MLB USA",                "lang": "en"},
    {"url": "https://www.mlssoccer.com/",                "league": "MLS USA",                "lang": "en"},
    {"url": "https://www.nba.com/",                      "league": "NBA USA",                "lang": "en"},
    {"url": "https://www.nfl.com/",                      "league": "NFL USA",                "lang": "en"},
    {"url": "https://www.nhl.com/",                      "league": "NHL USA/Canada",         "lang": "en"},
    {"url": "https://www.ligamx.net/",                   "league": "Liga MX Mexico",         "lang": "es"},
    {"url": "https://www.cbf.com.br/",                   "league": "CBF Brazil",             "lang": "pt"},
    {"url": "https://www.premierleague.com/",            "league": "Premier League UK",      "lang": "en"},
    {"url": "https://www.bundesliga.com/en/bundesliga/", "league": "Bundesliga Germany",     "lang": "de"},
    {"url": "https://www.bundesliga.at/de",              "league": "Bundesliga Austria",     "lang": "de"},
    {"url": "https://www.laliga.com/en-GB",              "league": "La Liga Spain",          "lang": "es"},
    {"url": "https://www.legaseriea.it/it",              "league": "Serie A Italy",          "lang": "it"},
    {"url": "https://ligue1.com/",                       "league": "Ligue 1 France",         "lang": "fr"},
    {"url": "https://top14.lnr.fr/",                    "league": "Top 14 Rugby France",    "lang": "fr"},
    {"url": "https://www.efl.com/",                      "league": "EFL England",            "lang": "en"},
    {"url": "https://eredivisie.nl/",                   "league": "Eredivisie Netherlands",  "lang": "nl"},
    {"url": "https://www.ligaportugal.pt/",              "league": "Liga Portugal",          "lang": "pt"},
    {"url": "https://sfl.ch/",                           "league": "Swiss Football League",  "lang": "de"},
    {"url": "https://www.tff.org/",                      "league": "TFF Turkey",             "lang": "tr"},
    {"url": "https://www.afl.com.au/",                   "league": "AFL Australia",          "lang": "en"},
    {"url": "https://www.nrl.com/",                      "league": "NRL Australia",          "lang": "en"},
    {"url": "http://eng.koreabaseball.com/",             "league": "KBO Korea",              "lang": "ko"},
    {"url": "https://www.jleague.jp/en/",                "league": "J-League Japan",         "lang": "ja"},
    {"url": "https://www.iplt20.com/",                   "league": "IPL India",              "lang": "en"},
    {"url": "https://www.indiansuperleague.com/",        "league": "ISL India",              "lang": "en"},
    {"url": "https://sports.sina.com.cn/csl/",           "league": "CSL China",              "lang": "zh"},
    {"url": "https://uscenterforsafesport.org/",         "league": "US Center Safe Sport",   "lang": "en"},
]

# ── FINANCE SITES ─────────────────────────────────────
FINANCE_SITES = [
    {"url": "https://www.nyse.com/index",                "exchange": "NYSE USA",             "lang": "en"},
    {"url": "https://www.nasdaq.com/",                   "exchange": "NASDAQ USA",           "lang": "en"},
    {"url": "https://www.cmegroup.com/",                 "exchange": "CME Group USA",        "lang": "en"},
    {"url": "https://www.cboe.com/",                     "exchange": "CBOE USA",             "lang": "en"},
    {"url": "https://www.nadex.com/",                    "exchange": "NADEX USA",            "lang": "en"},
    {"url": "https://www.tmx.com/",                      "exchange": "TMX Canada",           "lang": "en"},
    {"url": "https://www.m-x.ca/en/",                   "exchange": "MX Canada",            "lang": "en"},
    {"url": "https://www.bcr.com.ar/eng/default.aspx",   "exchange": "BCR Argentina",        "lang": "es"},
    {"url": "https://www.euronext.com/en",               "exchange": "Euronext Europe",      "lang": "en"},
    {"url": "https://www.eurex.com/ex-en/",              "exchange": "Eurex Germany",        "lang": "en"},
    {"url": "https://deutsche-boerse.com/dbg-en/",       "exchange": "Deutsche Boerse",      "lang": "de"},
    {"url": "https://www.lseg.com/en",                   "exchange": "LSEG London",          "lang": "en"},
    {"url": "https://www.lme.com/",                      "exchange": "LME London",           "lang": "en"},
    {"url": "https://www.six-group.com/en/",             "exchange": "SIX Swiss Exchange",   "lang": "de"},
    {"url": "https://www.nordpoolgroup.com/",            "exchange": "Nord Pool Energy",     "lang": "en"},
    {"url": "https://www.epexspot.com/en",               "exchange": "EPEX Spot Energy",     "lang": "en"},
    {"url": "https://www.bolsasymercados.es/ing/Home",   "exchange": "BME Spain",            "lang": "es"},
    {"url": "https://www.dgcx.ae/",                      "exchange": "DGCX Dubai",           "lang": "en"},
    {"url": "https://www.gulfmerc.com/",                 "exchange": "Gulf Mercantile",      "lang": "en"},
    {"url": "https://www.nasdaqdubai.com/",              "exchange": "NASDAQ Dubai",         "lang": "en"},
    {"url": "https://www.jse.co.za/trade",               "exchange": "JSE South Africa",     "lang": "en"},
    {"url": "https://www.ea-africaexchange.com/",        "exchange": "EA Africa Exchange",   "lang": "en"},
    {"url": "https://www.ecx.com.et/",                   "exchange": "ECX Ethiopia",         "lang": "en"},
    {"url": "https://www.hkex.com.hk/",                  "exchange": "HKEX Hong Kong",       "lang": "zh"},
    {"url": "https://www.jpx.co.jp/english/",            "exchange": "JPX Japan",            "lang": "ja"},
    {"url": "https://www.tfx.co.jp/en/",                 "exchange": "TFX Japan",            "lang": "ja"},
    {"url": "https://www.sse.com.cn/",                   "exchange": "SSE Shanghai",         "lang": "zh"},
    {"url": "https://www.szse.cn/English/",              "exchange": "SZSE Shenzhen",        "lang": "zh"},
    {"url": "http://www.cffex.com.cn/",                  "exchange": "CFFEX China",          "lang": "zh"},
    {"url": "http://www.dce.com.cn/",                    "exchange": "DCE Dalian China",     "lang": "zh"},
    {"url": "https://www.bseindia.com/",                 "exchange": "BSE India",            "lang": "en"},
    {"url": "https://www.nseindia.com/",                 "exchange": "NSE India",            "lang": "en"},
    {"url": "https://www.mcxindia.com/",                 "exchange": "MCX India",            "lang": "en"},
    {"url": "https://global.krx.co.kr/",                 "exchange": "KRX Korea",            "lang": "ko"},
    {"url": "https://www.sgx.com/",                      "exchange": "SGX Singapore",        "lang": "en"},
    {"url": "https://www.twse.com.tw/en/",               "exchange": "TWSE Taiwan",          "lang": "zh"},
    {"url": "https://mce.mn/",                           "exchange": "MCE Mongolia",         "lang": "mn"},
    {"url": "https://www.miaxglobal.com/",               "exchange": "MIAX USA",             "lang": "en"},
    {"url": "https://www.bcr.com.ar/eng/default.aspx/",  "exchange": "BCR Argentina",        "lang": "en"},
    {"url": "http://www.shfe.com.cn/en/",                "exchange": "SHFE Shanghai",        "lang": "en"},
    {"url": "http://www.czce.com.cn/en/",                "exchange": "CZCE Zhengzhou",       "lang": "en"},
    {"url": "https://www.ncdex.com/markets/",            "exchange": "NCDEX India",          "lang": "en"},
]

# ── SCRAPER ───────────────────────────────────────────
def scrape_site(site_info):
    try:
        headers  = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(site_info["url"], timeout=15, headers=headers)
        soup     = BeautifulSoup(response.text, "html.parser")
        text     = " ".join([p.get_text() for p in soup.find_all("p")[:30]])
        return {
            "url":       site_info["url"],
            "name":      site_info.get("league") or site_info.get("exchange"),
            "lang":      site_info["lang"],
            "content":   text[:3000],
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "status":    "success"
        }
    except Exception as e:
        return {
            "url":       site_info["url"],
            "name":      site_info.get("league") or site_info.get("exchange"),
            "lang":      site_info["lang"],
            "content":   "",
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "status":    f"error: {str(e)}"
        }

# ── REPORT GENERATOR ──────────────────────────────────
def generate_report(docs, category):
    context = ""
    for doc in docs[:10]:
        if doc["status"] == "success":
            context += f"\n[{doc['name']}]: {doc['content'][:400]}\n"

    prompt = f"""
    You are a Senior {category} Analyst.
    Write a concise professional report based ONLY on this data:
    {context}

    ## {category} Report — {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')} UTC
    ### KEY HIGHLIGHTS
    ### DETAILED ANALYSIS
    ### SOURCES CITED
    *Disclaimer: Informational purposes only. Not financial advice.*
    """

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="gemini-1.5-flash",
                contents=prompt
            )
            return response.text
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                wait = 60 * (attempt + 1)
                print(f"  ⏳ Rate limit — waiting {wait}s (attempt {attempt+1}/3)...")
                time.sleep(wait)
            else:
                return f"[Error: {str(e)}]"
    return "[Failed after 3 attempts]"

# ── EMAIL SENDER ──────────────────────────────────────
def send_email(to_email, subject, html_body):
    msg = MIMEMultipart("alternative")
    msg["Subject"] = subject
    msg["From"]    = SENDER_EMAIL
    msg["To"]      = to_email
    msg.attach(MIMEText(html_body, "html"))
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(SENDER_EMAIL, SENDER_PASSWORD)
            server.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        print(f"  ✅ Delivered to: {to_email}")
    except Exception as e:
        print(f"  ❌ Failed: {e}")

# ── REPORT DELIVERY ───────────────────────────────────
def deliver_reports(sports_report, finance_report):
    if not SUBSCRIBERS:
        print("  ℹ️  No subscribers yet.")
        return
    for email, name, reports_per_day in SUBSCRIBERS:
        price_monthly = reports_per_day * 10
        price_yearly  = price_monthly * 12
        html = f"""
        <html><body style="font-family:Arial,sans-serif;max-width:700px;margin:auto;">
        <div style="background:#1a1a2e;color:white;padding:20px;border-radius:10px;">
            <h2>📊 AI Intelligence Report</h2>
            <p>{datetime.now(timezone.utc).strftime('%A, %B %d, %Y — %H:%M UTC')}</p>
        </div>
        <div style="padding:20px;background:#e8f8e8;border-radius:10px;margin-top:10px;">
            <h2>🏆 SPORTS REPORT</h2>
            <pre style="white-space:pre-wrap;font-family:Arial;font-size:13px;">{sports_report}</pre>
        </div>
        <div style="padding:20px;background:#e8f0f8;border-radius:10px;margin-top:10px;">
            <h2>💰 FINANCE REPORT</h2>
            <pre style="white-space:pre-wrap;font-family:Arial;font-size:13px;">{finance_report}</pre>
        </div>
        <div style="padding:15px;background:#fff3cd;border-radius:10px;margin-top:10px;">
            <h3>💳 Subscribe / Upgrade</h3>
            <p>Monthly (${price_monthly}/mo):
               <a href="{STRIPE_MONTHLY}">{STRIPE_MONTHLY}</a></p>
            <p>Annual (${price_yearly}/yr):
               <a href="{STRIPE_YEARLY}">{STRIPE_YEARLY}</a></p>
        </div>
        <div style="padding:15px;text-align:center;color:#999;font-size:11px;">
            <p>Subscriber: {name} | {reports_per_day} reports/day</p>
            <p>📧 {CONTACT_EMAIL_1} | {CONTACT_EMAIL_2}</p>
            <p><em>Informational purposes only. Not financial advice.</em></p>
            <p>Reply UNSUBSCRIBE to cancel.</p>
        </div>
        </body></html>
        """
        send_email(
            email,
            f"📊 AI Report — {datetime.now(timezone.utc).strftime('%b %d %H:%M')} UTC",
            html
        )

# ── FULL PIPELINE ─────────────────────────────────────
def run_pipeline():
    now = datetime.now(timezone.utc)
    print(f"\n{'='*55}")
    print(f"🔄 PIPELINE START — {now.strftime('%Y-%m-%d %H:%M:%S')} UTC")
    print(f"{'='*55}")

    # 1. Scrape
    print("\n📡 Scraping all 71 sites...")
    all_docs = []
    for i, site in enumerate(SPORTS_SITES + FINANCE_SITES):
        result = scrape_site(site)
        all_docs.append(result)
        icon = "✅" if result["status"] == "success" else "⚠️"
        print(f"  {icon} [{i+1}/71] {result['name']}")
        time.sleep(1)

    sports_docs  = all_docs[:len(SPORTS_SITES)]
    finance_docs = all_docs[len(SPORTS_SITES):]
    ok = len([d for d in all_docs if d["status"] == "success"])
    print(f"\n  ✅ Scraped: {ok}/71 | ⚠️ Errors: {71-ok}/71")

    # 2. Generate reports
    print("\n📝 Generating reports with Gemini AI...")
    sports_report  = generate_report(sports_docs,  "Sports")
    finance_report = generate_report(finance_docs, "Financial")
    print("  ✅ Sports report ready")
    print("  ✅ Finance report ready")

    # 3. Deliver
    print(f"\n📧 Delivering to {len(SUBSCRIBERS)} subscriber(s)...")
    deliver_reports(sports_report, finance_report)

    print(f"\n✅ PIPELINE COMPLETE — {datetime.now(timezone.utc).strftime('%H:%M')} UTC")
    print(f"{'='*55}")

# ── STARTUP CONFIRMATION ──────────────────────────────
print(f"✅ Sports sites:      {len(SPORTS_SITES)}")
print(f"✅ Finance sites:     {len(FINANCE_SITES)}")
print(f"✅ Total KB:          {len(SPORTS_SITES)+len(FINANCE_SITES)} sites")
print(f"✅ Subscribers:       {len(SUBSCRIBERS)}")
print(f"✅ Credentials:       Loaded from Colab Secrets 🔐")
print(f"\n{'='*55}")
print(f"🚀 SYSTEM READY — Running pipeline now...")
print(f"{'='*55}\n")

# ── AUTO-RUN PIPELINE ─────────────────────────────────
run_pipeline()


✅ google-genai installed
⏳ Loading Complete System...
✅ Sports sites:      30
✅ Finance sites:     42
✅ Total KB:          72 sites
✅ Subscribers:       1
✅ Credentials:       Loaded from Colab Secrets 🔐

🚀 SYSTEM READY — Running pipeline now...


🔄 PIPELINE START — 2026-03-30 14:08:15 UTC

📡 Scraping all 71 sites...
  ✅ [1/71] AFA Argentina
  ✅ [2/71] Dimayor Colombia
  ✅ [3/71] CFL Canada
  ✅ [4/71] MLB USA
  ✅ [5/71] MLS USA
  ✅ [6/71] NBA USA
  ✅ [7/71] NFL USA
  ✅ [8/71] NHL USA/Canada
  ✅ [9/71] Liga MX Mexico
  ✅ [10/71] CBF Brazil
  ✅ [11/71] Premier League UK
  ✅ [12/71] Bundesliga Germany
  ✅ [13/71] Bundesliga Austria
  ✅ [14/71] La Liga Spain
  ✅ [15/71] Serie A Italy
  ✅ [16/71] Ligue 1 France
  ✅ [17/71] Top 14 Rugby France
  ✅ [18/71] EFL England
  ✅ [19/71] Eredivisie Netherlands
  ✅ [20/71] Liga Portugal
  ✅ [21/71] Swiss Football League
  ✅ [22/71] TFF Turkey
  ✅ [23/71] AFL Australia
  ✅ [24/71] NRL Australia
  ✅ [25/71] KBO Korea
  ✅ [26/71] J-League Japan
  ✅ [27/7

# New Section